# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Estrategia de validación cruzada anidada (Nested Cross Validation): Justificación e implementación

### 📝 Descripción del notebook
En este notebook se presenta la estrategia de validación cruzada *Nested Cross Validation* utilizada para evaluar de forma robusta el rendimiento de distintos modelos de clasificación. Se explican las ventajas de esta metodología frente a otras estrategias de validación, así como el procedimiento empleado para garantizar la reproducibilidad de los experimentos mediante la generación y almacenamiento de los folds en archivos CSV.

### Indice de contenidos
1. [Nested Cross Validation](#nested_cross_validation)
    * [1.1 Limitaciones de los enfoques tradicionales](#limitaciones)
    * [1.2 Estrategia de validación cruzada anidada (Nested Cross Validation)](#explicacion)
        * [1.2.1 Formulación matemática](#formulacion_matematica)
        * [1.2.2 Interpretación del rendimiento](#interpretacion)
    * [1.3 Implementación práctica en este trabajo](#implementacion)    
 
2. [Procedimiento de generación de los folds (reproducibilidad)](#reproducibilidad)
    * [2.1 Formato de los ficheros CSV](#formato_ficheros_csv)
        * [2.1.1 Fichero para Outer CV: *outer_folds.csv*](#outer_folds_csv)
        * [2.1.2 Ficheros para Inner CV: *inner_fold_{outer_fold_idx}.csv*](#inner_folds_csv)    

3. [Validación de los folds creados](#validacion)

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero jupyter notebook, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [2]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

In [3]:
import warnings
import random
import numpy as np
import pandas as pd

# Configuration object
from src.config import cfg

# Nested Cross Validation methods
from src.nested_cv_utils import generate_folds, validate_folds

**Import troubleshooting**

If the `src` imports fail when running this notebook in a different environment,
uncomment and execute the following cell:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
```

In [4]:
# Global configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

In [5]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

<a id="nested_cross_validation"></a>
# 1. Nested Cross Validation

En esta sección se introduce la estrategia de validación cruzada *Nested Cross Validation* planteada en el artículo: *Varma, S., & Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. BMC Bioinformatics, 7(1), 1-8*.

<a id="limitaciones"></a>
## 1.1 Limitaciones de los enfoques tradicionales

Los métodos de validación tradicionales presentan limitaciones importantes, especialmente cuando se aplican a conjuntos de datos con un número reducido de muestras.

El enfoque clásico basado en el método Hold-Out presenta un problema fundamental en este contexto. Para obtener una estimación fiable del error de generalización del modelo, el conjunto de test debe contener un número suficientemente representativo de muestras. Cuando el conjunto de test es reducido, se incrementa la varianza de la estimación del error y esto puede producir evaluaciones poco fiables.

Además, al disponer de pocos datos, realizar una única partición en conjuntos de entrenamiento y test introduce una fuerte dependencia respecto a la selección concreta de las muestras incluidas en cada subconjunto. Como consecuencia, el rendimiento estimado puede variar significativamente dependiendo de la partición utilizada, ya que una única división difícilmente captura toda la variabilidad presente en el conjunto de datos.

Para mitigar estas limitaciones aparece la validación cruzada tradicional (*Cross-Validation*). Esta estrategia permite aprovechar de forma más eficiente los datos disponibles, ya que cada muestra participa tanto en el entrenamiento como en la evaluación del modelo en distintas iteraciones. De esta manera, la validación cruzada proporciona una estimación más robusta del rendimiento de un modelo, con hiperparámetros fijos, sobre datos no vistos.

Sin embargo, una práctica habitual consiste en utilizar el mismo procedimiento de validación cruzada tanto para la optimización del modelo (selección de hiperparámetros) como para la evaluación final de su rendimiento. Cuando el mejor resultado obtenido durante la búsqueda de hiperparámetros se reporta directamente como estimación del error de generalización, la evaluación resultante queda sesgada de forma optimista.

Este problema aparece porque los hiperparámetros son seleccionados específicamente para maximizar el rendimiento sobre los mismos datos utilizados durante la validación. Como consecuencia, el modelo termina adaptándose parcialmente al conjunto de evaluación, produciendo un sesgo de selección de hiperparámetros y una subestimación del error real de generalización. En el trabajo de *Rajeev Varma y Richard Simon* se demuestra que este sesgo tiende a producir estimaciones excesivamente optimistas del rendimiento, especialmente en conjuntos de datos pequeños.

Para solucionar este error metodológico se reserva una única partición de test para evaluar el modelo, después de haber realizado la selección de hiperparámetros mediante validación cruzada. Esta práctica evita el sesgo introducido por utilizar los mismos datos durante la optimización y la evaluación del modelo.

Sin embargo, esta forma de proceder reintroduce el problema asociado a conjuntos de datos pequeños en los que una única partición de test no refleja toda la variabilidad de los datos. En este contexto, la evaluación del rendimiento puede estar muy sesgada hacia la partición concreta que se ha realizado.

Para aprovechar todos los datos disponibles y, al mismo tiempo, obtener una evaluación no sesgada del rendimiento, se emplea la estrategia de validación cruzada anidada (Nested Cross Validation) planteada en *Varma, S., & Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. BMC Bioinformatics, 7(1), 1-8*

<a id="explicacion"></a>
## 1.2 Estrategia de validación cruzada anidada (Nested Cross Validation)

Este enfoque de validación separa completamente la optimización del modelo (selección de hiperparámetros) de la evaluación de su rendimiento, lo que resulta en una estimación muy cercana a la que se obtendría con un conjunto de test independiente. La estrategia consiste en implementar dos bucles de validación cruzada uno dentro del otro:
* **Validación cruzada externa (*Outer CV Loop*)**: utilizada para evaluar el rendimiento del modelo en datos que no han tenido ninguna influencia en la elección de los hiperparámetros.
* **Validación cruzada interna (*Inner CV Loop*)**: utilizada para realizar la optimización del modelo (selección de hiperparámetros) únicamente sobre datos de entrenamiento.

<a id="formulacion_matematica"></a>
### 1.2.1 Formulación matemática

Sea $D$ el conjunto de datos total, se definen dos procesos de particionamiento indexados por un bucle externo y un bucle interno. En cada bucle se realiza un proceso de validación cruzada independiente.

**Bucle externo (*Outer Loop*)**

El conjunto de datos $D$ se particiona aleatoriamente en $K_{out}$ subconjuntos disjuntos de tamaño similar, tales que:
$$D = \bigcup_{i=1}^{K_{out}} D_i^{out} \quad \text{donde} \quad D_i^{out} \cap D_j^{out} = \emptyset \text{ para } i \neq j$$

El número $K_{out}$ de subconjuntos representa el número de *folds* correspondientes a la validación cruzada externa. Para cada *fold* $i$, con $i \in \{1, 2, \dots, K_{out}\}$, se definen dos subconjuntos:
* **Outer Test Set**: Se trata del conjunto de *test* de la validación cruzada externa, denotado como $D_i^{out}$.
* **Outer Train Set**: Se trata del conjunto de entrenamiento de la validación cruzada externa. Está compuesto por los $K_{out}-1$ subconjuntos restantes (excluyendo $D_i^{out}$). En términos de conjuntos, puede expresarse como el complemento del *Outer Test Set*:
$$D_{-i}^{out} = D \setminus D_i^{out}$$

**Bucle interno (*Inner Loop*)**

Para cada iteración $i$ del bucle externo, se ejecuta un proceso de optimización sobre el *Outer Train Set* ($D_{-i}^{out}$) para encontrar la configuración óptima de hiperparámetros. Este proceso es implementado mediante una validación cruzada interna sobre este conjunto de datos.

El *Outer Train Set* es dividido en $K_{in}$ subconjuntos disjuntos:
$$D_{-i}^{out} = \bigcup_{j=1}^{K_{in}} D_{ij}^{in} \quad \text{donde} \quad D_{ij}^{in} \cap D_{ik}^{in} = \emptyset \text{ para } j \neq k$$

El número $K_{in}$ de subconjuntos representa el número de *folds* correspondientes a la validación cruzada interna. Para cada *fold* $j$, con $j \in \{1, 2, \dots, K_{in}\}$, se definen dos subconjuntos:
* **Inner Validation Set**: Se trata del conjunto de validación de la validación cruzada interna, denotado como $D_{ij}^{in}$.
* **Inner Train Set**: Se trata del conjunto de entrenamiento de la validación cruzada interna. Está compuesto por los $K_{in}-1$ subconjuntos restantes (excluyendo $D_{ij}^{in}$). En términos de conjuntos, puede expresarse como el complemento del *Inner Validation Set*:
$$D_{i,-j}^{in} = D_{-i}^{out} \setminus D_{ij}^{in}$$

En cada iteración de la validación cruzada interna, se evalúa el rendimiento de todas las combinaciones de hiperparámetros usando una métrica objetivo. Finalmente, se selecciona la configuración de hiperparámetros que, en promedio, ha maximizado o minimizado dicha métrica.

**Evaluación final**

Tras finalizar el procedimiento de validación cruzada interna (*Inner Loop*), se obtiene la configuración de hiperparámetros óptima para el *Outer Train Set* de la iteración externa $i$. A continuación se procede de la siguiente forma:

* El modelo se entrena utilizando la configuración óptima sobre todo el conjunto de datos *Outer Train Set* ($D_{-i}^{out}$).
* Se evalúa dicho modelo sobre el conjunto de datos *Outer Test Set* ($D_i^{out}$).

Este procedimiento se repite para las $K_{out}$ particiones del bucle externo, por lo que se obtienen $K_{out}$ evaluaciones del rendimiento independientes.

<a id="interpretacion"></a>
### 1.2.2 Interpretación del rendimiento 

La estrategia *Nested Cross Validation* ofrece una estimación del rendimiento que representa, de forma robusta y realista, el comportamiento del modelo ante datos completamente nuevos. A diferencia de los métodos tradicionales, su valor reside en evaluar el proceso de aprendizaje en sí mismo, no un único modelo final estático. De esta forma, proporciona una idea de cómo de bien un determinado algoritmo es capaz de aprender de un conjunto de entrenamiento y generalizar ante un conjunto de datos nuevos.

Al finalizar esta validación cruzada, se obtienen $K_{out}$ evaluaciones de rendimiento (métricas) independientes que se interpretan mediante su valor promedio y su varianza (o desviación estándar).

Este enfoque es especialmente útil cuando se trabaja con un conjunto de datos pequeño. En lugar de congelar un $20\%$ o $30\%$ de los datos para evaluación, la totalidad de los datos son utilizados tanto para entrenar como para evaluar en diferentes momentos del proceso. Por lo tanto, el rendimiento promedio está respaldado por todo el *dataset* y no depende de una única partición fija, la cual podría presentar una alta varianza debido a la escasez de datos.

<a id="implementacion"></a>
## 1.3 Implementación práctica en este trabajo

La implementación práctica de la validación cruzada anidada (*Nested Cross Validation*) requiere definir el número de particiones (*folds*) tanto para el bucle externo ($K_{out}$) como para el bucle interno ($K_{in}$).

El objetivo de este trabajo es evaluar el rendimiento de cinco familias de modelos con diferentes naturalezas algorítmicas y niveles de complejidad: Regresión Logística Multinomial, Árboles de Decisión, Random Forest, Máquinas de Vectores de Soporte (SVM) con kernel RBF y XGBoost.

Bajo el esquema de Nested Cross Validation, el número total de entrenamientos necesarios para un algoritmo con $H$ combinaciones de hiperparámetros viene dado por:
$$\text{Total entrenamientos} = (K_{out}\times K_{in} \times H) + K_{out} = K_{out} \times (K_{in} \times H + 1)$$
donde el término $(K_{out} \times K_{in} \times H)$ corresponde al proceso de búsqueda de hiperparámetros en el bucle interno, y $K_{out}$ al entrenamiento final del modelo óptimo sobre el *Outer Train Set*.

En este trabajo se ha seleccionado una configuración de $K_{out} = 5$ folds para el bucle externo y $K_{in} = 3$ folds para el bucle interno. Esta elección responde a un compromiso entre la robustez estadística de los resultados y el coste computacional del experimento, teniendo en cuenta además la naturaleza del conjunto de datos, caracterizado por un tamaño reducido y un cierto desbalanceo de clases.

En cada iteración del bucle externo, el conjunto de datos se divide en 5 particiones, utilizando aproximadamente el $80\%$ de las muestras para el entrenamiento externo (*Outer Train Set*) y el $20\%$ restante para la evaluación (*Outer Test Set*). A su vez, el *Outer Train Set* se somete a una validación cruzada interna de 3 folds, en la que cada partición utiliza aproximadamente un $66.7\%$ de los datos para entrenamiento interno (*Inner Train Set*) y un $33.3\%$ para validación (*Inner Validation Set*), con el objetivo de optimizar los hiperparámetros del modelo.

Ambas particiones, interna y externa, se realizan de forma estratificada (*Stratified K-Fold*), garantizando la preservación de la distribución original de las clases en cada subconjunto.
´
La configuración del bucle interno no solo responde a consideraciones computacionales, sino también a la necesidad de mantener una representación adecuada de las clases minoritarias durante la optimización de hiperparámetros. Dado que el *Outer Train Set* representa únicamente una fracción del conjunto de datos original, el uso de un número excesivo de folds podría dar lugar a particiones con un número insuficiente de muestras en las clases menos representadas. En este sentido, la elección de $K_{in} = 3$ junto con la estratificación, contribuye a mantener un equilibrio adecuado entre representatividad de clases y estabilidad en la estimación del rendimiento.

A continuación se ilustra de forma gráfica el esquema de validación cruzada anidada utilizado en este trabajo.

![Esquema de Validación Cruzada Anidada](../images/Nested_cross_validation.png)

<a id="reproducibilidad"></a>
# 2. Procedimiento de generación de los folds (reproducibilidad)

Para garantizar la reproducibilidad de los experimentos, los folds utilizados en la validación cruzada han sido generados previamente y almacenados en ficheros CSV. Este enfoque permite asegurar que todas las ejecuciones del experimento utilicen exactamente las mismas particiones de datos, facilitando la comparación directa entre modelos.

La generación de estas particiones se ha implementado mediante la función *generate_folds()*, incluida en el módulo *nested_cv_fold.py* (ubicado en el directorio *src/*). Esta función construye los folds a partir de los parámetros definidos en el archivo de configuración *config.py*, incluyendo el número de particiones y la semilla aleatoria, y los almacena en el directorio *data/folds*.

Se ha utilizado la siguiente configuración experimental (definida en config.py):
* *OUTER_SPLITS* = 5.
* *INNER_SPLITS* = 3.
* *SEED* = 42.

A continuación se muestra la ejecución de dicha función, lo que genera y almacena los *folds*.

In [6]:
generate_folds(cfg)

Generating outer fold 0

Generating inner fold 0 for outer fold 0
Generating inner fold 1 for outer fold 0
Generating inner fold 2 for outer fold 0
Save complete inner fold information for outer fold 0 in inner_fold_0.csv
------------------------------


Generating outer fold 1

Generating inner fold 0 for outer fold 1
Generating inner fold 1 for outer fold 1
Generating inner fold 2 for outer fold 1
Save complete inner fold information for outer fold 1 in inner_fold_1.csv
------------------------------


Generating outer fold 2

Generating inner fold 0 for outer fold 2
Generating inner fold 1 for outer fold 2
Generating inner fold 2 for outer fold 2
Save complete inner fold information for outer fold 2 in inner_fold_2.csv
------------------------------


Generating outer fold 3

Generating inner fold 0 for outer fold 3
Generating inner fold 1 for outer fold 3
Generating inner fold 2 for outer fold 3
Save complete inner fold information for outer fold 3 in inner_fold_3.csv
-------------

<a id="formato_ficheros_csv"></a>
## 2.1 Formato de los ficheros CSV

La función genera dos tipos de ficheros CSV en el directorio de salida *data/folds*. Estos archivos no contienen directamente las variables del conjunto de datos, sino que actúan como estructuras de mapeo que definen la asignación de cada muestra a los distintos *folds* mediante el identificador único **CC** que contiene el código de complejo residencial.

<a id="outer_folds_csv"></a>
### 2.1.1 Fichero para Outer CV: *outer_folds.csv*

Este archivo contiene la asignación de folds para la validación cruzada externa (Outer Cross Validation). Su estructura se compone de las siguientes columnas:
* **CC**. Identificador único del complejo residencial, correspondiente al valor de la variable CC en el conjunto de datos.
* **outer_fold_idx**. Indice del *fold* que indica que dicha muestra pertenece al conjunto de test de ese *fold*. Dado que se utilizan $5$ folds los indices pueden tomar valores en $[0,1,2,3,4]$

A continuación se muestran las cinco primeras filas de este archivo:

In [7]:
outer_df = pd.read_csv(cfg.DATA_FOLDS_DIR/"outer_folds.csv")
outer_df.head()

,CC,outer_fold_idx
0,301015,0
1,301019,0
2,301020,0
3,301012,0
4,301028,0


De este modo, la identificación de los conjuntos de entrenamiento y test para cada fold se realiza a través de la variable *outer_fold_idx*. Por ejemplo, para la primera partición asociada al fold $0$:
* **Conjunto de entrenamiento (*Outer Train Set*)**: todas las muestras cuyo *outer_fold_idx* es distinto de 0.
* **Conjunto de test (*Outer Test Set*)**: todas las muestras cuyo *outer_fold_idx* es igual a 0.


In [8]:
train_outer_df = outer_df[outer_df["outer_fold_idx"] != 0].copy()
test_outer_df = outer_df[outer_df["outer_fold_idx"] == 0].copy()
print("Número de muestras en el primer fold de Outer Cross Validation (fold 0):")
print(f"Conjunto de entrenamiento: {train_outer_df.shape[0]} muestras.")
print(f"Conjunto de test: {test_outer_df.shape[0]} muestras.")

Número de muestras en el primer fold de Outer Cross Validation (fold 0):
Conjunto de entrenamiento: 513 muestras.
Conjunto de test: 129 muestras.


<a id="inner_folds_csv"></a>
### 2.1.2 Ficheros para Inner CV: *inner_fold_{outer_fold_idx}.csv*

Para cada partición de la validación cruzada externa (*Outer CV*) se genera un fichero independiente que contiene la asignación de folds correspondiente a la validación cruzada interna (*Inner CV*). Dado que la validación externa se compone de $5$ folds, se generan en total cinco ficheros asociados a la validación interna:
* inner_fold_0.csv.
* inner_fold_1.csv.
* inner_fold_2.csv.
* inner_fold_3.csv.
* inner_fold_4.csv.

Cada uno de estos ficheros contiene las siguientes columnas:
* **CC**. Identificador único del complejo residencial que se corresponda con el valor de la variable **CC** en el conjunto de datos.
* **inner_fold_idx**. Indice del *fold* que indica que dicha muestra pertenece al conjunto de validación de ese *fold*. Dado que se utilizan $3$ *folds* los indices pueden tomar valores en $[0,1,2]$.

A continuación se muestran las cinco primeras filas del fichero *inner_fold_0.csv* asociado al fold $0$ de la validación externa.

In [9]:
inner_df = pd.read_csv(cfg.DATA_FOLDS_DIR/"inner_fold_0.csv")
inner_df.head()

,CC,inner_fold_idx
0,301014,0
1,301016,0
2,301027,0
3,2201023,0
4,8709041,0


Al igual que en el caso anterior, la variable *inner_fold_idx* se utiliza para definir los conjuntos de entrenamiento y validación dentro de cada partición. Por ejemplo, para la primera partición asociada al *Inner fold 0*:
* **Conjunto de entrenamiento (*Inner Train Set*)**: todas las muestras cuyo *inner_fold_idx* es distinto de 0.
* **Conjunto de test (*Inner Validation Set*)**: todas las muestras cuyo *inner_fold_idx* es igual a 0.

A continuación, se muestra el número de muestras correspondientes a los conjuntos de entrenamiento y validación del primer *inner fold*.

In [10]:
train_inner_df = inner_df[inner_df["inner_fold_idx"] != 0].copy()
test_inner_df = inner_df[inner_df["inner_fold_idx"] == 0].copy()
print("Número de muestras en el primer fold de Outer Cross Validation (fold 0):")
print(f"Conjunto de entrenamiento: {train_outer_df.shape[0]} muestras.")
print(f"Conjunto de test: {test_outer_df.shape[0]} muestras.")

print("\nNúmero de muestras en el primer fold de la Inner Cross Validation asociada:")
print(f"Conjunto de entrenamiento: {train_inner_df.shape[0]}")
print(f"Conjunto de test: {test_inner_df.shape[0]}")

Número de muestras en el primer fold de Outer Cross Validation (fold 0):
Conjunto de entrenamiento: 513 muestras.
Conjunto de test: 129 muestras.

Número de muestras en el primer fold de la Inner Cross Validation asociada:
Conjunto de entrenamiento: 342
Conjunto de test: 171


Como se puede observar, las $513$ muestras del conjunto de entrenamiento (*Outer Train Set*) de la *Outer CV* se distribuyen en $342$ muestras para el conjunto de entrenamiento (*Inner Train Set*) y $171$ para el conjunto de validación (*Inner Validation Set*) en la primera partición de la *Inner CV*.

<a id="validacion"></a>
# 3. Validación de los folds creados

Para garantizar que los *folds* generados con la función *generate_folds()* son correctos y respetan el esquema *Nested Cross Validation*, se ha desarrollado la función *validate_folds()* (en el módulo *nested_cv_fold.py*).

Para el fichero de la validación cruzada externa (*outer_folds.csv*) se verifica lo siguiente:
* Todas las muestras del *dataset* aparecen en el fichero.
* Cada muestra pertenece a un único *Outer fold* sin duplicidades.

Para cada *fold* de la validación cruzada externa, se verifica que los *inner folds* asociados cumplen:
* Las muestras presentes en el fichero *inner_fold_{outer_fold_idx}.csv*  pertenecen al conjunto de entrenamiento externo (*Outer Train Set*) asociado.
* Cada muestra pertenece a un único *Inner fold* sin duplicidades.

A continuación se muestra la ejecución de dicha función que no detecta ningún error en los ficheros generados.

In [11]:
validate_folds(cfg)


✅ All Nested CV folds are valid.
